<a href="https://colab.research.google.com/github/IFuentesSR/glaciers_assessment/blob/main/glaciers_gee.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import ee
ee.Authenticate()
ee.Initialize(project='upbeat-imprint-269809')  # replace with your own project if needed

/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


In [4]:
import math

In [3]:
# -----------------------------------------------------------------------------
# INPUTS
# -----------------------------------------------------------------------------
geometry = ee.Geometry.Polygon([[
    [-70.05593539819137, -33.69841007909303],
    [-70.05359512766677, -33.700468100972856],
    [-70.04642508760617, -33.695515070834574],
    [-70.04298518294603, -33.6932495306298],
    [-70.03954525437226, -33.69583995635748],
    [-70.03644858888146, -33.6947169217856],
    [-70.03848164834116, -33.68979617020842],
    [-70.03619632238691, -33.68716713777566],
    [-70.0356597987545, -33.68483706413488],
    [-70.03572418714073, -33.68114992454193],
    [-70.0391359207188, -33.681319551192246],
    [-70.03849221582382, -33.680158905851044],
    [-70.03862098386233, -33.67899825850726],
    [-70.03905013730471, -33.67756975371347],
    [-70.04291251828616, -33.678641134533784],
    [-70.04475785594873, -33.67999818745432],
    [-70.04574493100588, -33.68149808478749],
]])

L8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
L9 = ee.ImageCollection('LANDSAT/LC09/C02/T1_L2')
L5 = ee.ImageCollection('LANDSAT/LT05/C02/T1_L2')
L7 = ee.ImageCollection('LANDSAT/LE07/C02/T1_L2')

INV_ASSET = 'projects/lacoloradachilesebastianm1234/assets/INV_PG_2022_v2'
CLASES_OBJETIVO = [
    'GLACIAR DE VALLE',
    'GLACIAR DE MONTAÑA',
    'GLACIAR DE EFLUENTE',
    'GLACIAR VALLE',
]

aoi = geometry
TILE_SCALE = 16
START = '1984-01-01'
END = '2025-12-31'

# Glacier inventory subset used in the final analysis.
glaciars = (
    ee.FeatureCollection(INV_ASSET)
    .filter(ee.Filter.inList('CLASIFICA', CLASES_OBJETIVO))
    .filter(
        ee.Filter.Or(
            ee.Filter.eq('COD_REG', '05'),
            ee.Filter.eq('COD_REG', '13'),
        )
    )
)


In [5]:
# DEM and terrain variables.
dem = ee.Image('NASA/NASADEM_HGT/001').select('elevation')
terrain = ee.Terrain.products(dem)
slope = terrain.select('slope').multiply(math.pi / 180.0)
aspect = terrain.select('aspect').multiply(math.pi / 180.0)

In [6]:
def set_tiles(img):
    img = ee.Image(img)
    date_str = ee.Date(img.get('system:time_start')).format('YYYY-MM-dd')
    tile = ee.String(img.get('system:index')).slice(7, 13)
    return img.set({'date': date_str, 'tile': tile})


def conversion(img):
    """Scale/rename Landsat 5/7 Collection-2 L2 bands to the L8/9 naming scheme."""
    img = ee.Image(img)
    props = img.propertyNames()

    etm_sr = ['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B7']
    lst_band = ['ST_B6']
    oli_names = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']

    optical = img.select(etm_sr).multiply(0.0000275).add(-0.2)
    thermal = img.select(lst_band).multiply(0.00341802).add(149.0)

    return (
        img.addBands(optical.rename(oli_names), None, True)
        .addBands(thermal.rename('TB'))
        .copyProperties(img, props)
    )


def scale_sr(image):
    """Scale Landsat 8/9 Collection-2 L2 surface reflectance and ST."""
    image = ee.Image(image)
    optical = image.select('SR_B.*').multiply(0.0000275).add(-0.2)
    thermal = image.select('ST_B10').multiply(0.00341802).add(149.0)

    return (
        image.addBands(optical, None, True)
        .addBands(thermal.rename('TB'))
        .copyProperties(image, image.propertyNames())
    )


def safe_dict(d):
    return ee.Dictionary(ee.Algorithms.If(d, d, ee.Dictionary({})))


def dict_get_or(d, key, fallback):
    d = safe_dict(d)
    value = d.get(key)
    return ee.Number(ee.Algorithms.If(value, value, fallback))


def add_soft_cloud_mask(img, coverage_region):
    """QA cloud/shadow mask plus optical/thermal coverage diagnostics."""
    img = ee.Image(img)
    coverage_region = ee.Geometry(coverage_region)

    footprint_mask = (
        img.select('SR_B3')
        .mask()
        .gt(0)
        .unmask(0, False)
        .rename('footprintMask')
        .toFloat()
    )

    qa = img.select('QA_PIXEL')
    cloud_free = qa.bitwiseAnd(1 << 3).eq(0)
    shadow_free = qa.bitwiseAnd(1 << 4).eq(0)

    qa_ok = (
        cloud_free.And(shadow_free)
        .And(footprint_mask.eq(1))
        .unmask(0, False)
        .rename('qaOK')
        .toFloat()
    )

    soft_mask = qa_ok.rename('softMask').toFloat()

    lst_mask = (
        img.select('TB')
        .mask()
        .gt(0)
        .And(soft_mask.eq(1))
        .unmask(0, False)
        .rename('lstMask')
        .toFloat()
    )

    coverage_stack = ee.Image.cat([footprint_mask, soft_mask, lst_mask])

    coverage_stats = safe_dict(
        coverage_stack.reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=coverage_region,
            scale=30,
            crs=img.select('SR_B3').projection(),
            maxPixels=1e13,
            tileScale=TILE_SCALE,
        )
    )

    footprint_cover = dict_get_or(coverage_stats, 'footprintMask', 0)
    usable_cover = dict_get_or(coverage_stats, 'softMask', 0)
    lst_cover = dict_get_or(coverage_stats, 'lstMask', 0)

    clear_fraction_observed = ee.Number(
        ee.Algorithms.If(
            footprint_cover.gt(0),
            usable_cover.divide(footprint_cover),
            0,
        )
    )

    return (
        img.addBands([footprint_mask, qa_ok, soft_mask, lst_mask])
        .updateMask(soft_mask)
        .set(
            {
                'footprint_cover': footprint_cover,
                'clear_fraction_observed': clear_fraction_observed,
                'usable_cover': usable_cover,
                'lst_cover': lst_cover,
                'soft_cover': usable_cover,
                'coverage_region': 'glacier_polygon',
                'thermal_cloud_filter': 0,
                'reduce_scale': 30,
            }
        )
    )


def topo_shadow(image):
    image = ee.Image(image)
    sun_az = ee.Number(image.get('SUN_AZIMUTH'))
    sun_el = ee.Number(image.get('SUN_ELEVATION'))
    zen = ee.Number(90).subtract(sun_el)

    hs = ee.Terrain.hillShadow(dem, sun_az, zen, 5, True)
    shadow = hs.Not().rename('shadow').focal_max(3)
    lit = shadow.Not().rename('lit')

    return image.addBands([shadow, lit])


def add_topo_bands(image):
    image = ee.Image(image)

    sun_az = ee.Image.constant(
        ee.Number(image.get('SUN_AZIMUTH')).multiply(math.pi / 180.0)
    )
    sun_el = ee.Image.constant(ee.Number(image.get('SUN_ELEVATION')))
    zenith = ee.Image.constant(90).subtract(sun_el).multiply(math.pi / 180.0)

    cos_s = zenith.cos()
    cos_i = (
        slope.cos()
        .multiply(zenith.cos())
        .add(
            slope.sin()
            .multiply(zenith.sin())
            .multiply(sun_az.subtract(aspect).cos())
        )
    )

    return image.addBands(cos_i.rename('cosI')).addBands(cos_s.rename('cosS'))


def c_correction(image, correction_region):
    image = ee.Image(image)
    correction_region = ee.Geometry(correction_region)

    bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    cos_i = image.select('cosI')
    cos_s = image.select('cosS')

    def correct_band(band_name):
        band_name = ee.String(band_name)
        band = image.select([band_name])

        regression_mask = band.mask().And(cos_i.mask()).And(cos_i.gt(1e-6))

        fit = ee.Dictionary(
            cos_i.updateMask(regression_mask)
            .addBands(band.updateMask(regression_mask))
            .reduceRegion(
                reducer=ee.Reducer.linearFit(),
                geometry=correction_region,
                scale=30,
                bestEffort=True,
                maxPixels=1e13,
                tileScale=TILE_SCALE,
            )
        )

        raw_slope = fit.get('scale')
        raw_offset = fit.get('offset')

        regression_slope = ee.Number(ee.Algorithms.If(raw_slope, raw_slope, 0))
        regression_offset = ee.Number(ee.Algorithms.If(raw_offset, raw_offset, 0))

        has_slope = ee.Number(ee.Algorithms.If(raw_slope, 1, 0))
        has_offset = ee.Number(ee.Algorithms.If(raw_offset, 1, 0))
        slope_nonzero = regression_slope.abs().gt(1e-6)

        has_valid_fit = has_slope.multiply(has_offset).multiply(slope_nonzero).eq(1)
        safe_slope = ee.Number(ee.Algorithms.If(slope_nonzero, regression_slope, 1))
        c_value = regression_offset.divide(safe_slope)

        denominator = cos_i.add(c_value)
        safe_denominator = denominator.where(denominator.abs().lte(1e-6), 1)

        corrected_candidate = (
            band.multiply(cos_s.add(c_value))
            .divide(safe_denominator)
            .updateMask(denominator.abs().gt(1e-6))
            .unmask(band)
        )

        corrected = ee.Image(
            ee.Algorithms.If(has_valid_fit, corrected_candidate, band)
        ).rename(band_name.cat('_ccorr'))

        return corrected

    corrected = ee.ImageCollection.fromImages(ee.List(bands).map(correct_band)).toBands()
    # toBands() prefixes names; select/rename back to expected names.
    expected = [f'{b}_ccorr' for b in bands]
    corrected = corrected.rename(expected)
    return image.addBands(corrected)


def minnaert_correction(image, correction_region):
    image = ee.Image(image)
    correction_region = ee.Geometry(correction_region)

    bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    cos_i = image.select('cosI')
    cos_s = image.select('cosS')

    def correct_band(band_name):
        band_name = ee.String(band_name)
        band = image.select([band_name])

        regression_mask = (
            cos_i.gt(1e-6)
            .And(band.gt(0))
            .And(cos_i.mask())
            .And(band.mask())
        )

        log_cos_i = cos_i.updateMask(regression_mask).log()
        log_band = band.updateMask(regression_mask).log()

        fit = ee.Dictionary(
            log_cos_i.addBands(log_band).reduceRegion(
                reducer=ee.Reducer.linearFit(),
                geometry=correction_region,
                scale=30,
                bestEffort=True,
                maxPixels=1e13,
                tileScale=TILE_SCALE,
            )
        )

        scale_value = fit.get('scale')
        has_valid_scale = ee.Number(ee.Algorithms.If(scale_value, 1, 0))
        k_raw = ee.Number(
            ee.Algorithms.If(has_valid_scale.eq(1), scale_value, 0)
        )
        k = k_raw.clamp(0, 2)

        denominator_valid = cos_i.gt(1e-6)
        ratio = cos_s.divide(cos_i.where(denominator_valid.Not(), 1))

        corrected_candidate = (
            band.multiply(ratio.pow(ee.Image.constant(k)))
            .updateMask(denominator_valid)
            .unmask(band)
        )

        corrected = ee.Image(
            ee.Algorithms.If(
                has_valid_scale.eq(1),
                corrected_candidate,
                band,
            )
        ).rename(band_name.cat('_ccorr'))

        return corrected

    corrected = ee.ImageCollection.fromImages(ee.List(bands).map(correct_band)).toBands()
    expected = [f'{b}_ccorr' for b in bands]
    corrected = corrected.rename(expected)
    return image.addBands(corrected)


def hybrid_topo_correction(image, correction_region):
    image = ee.Image(image)
    correction_region = ee.Geometry(correction_region)

    bands = ['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7']
    slope_deg = ee.Terrain.slope(dem)

    ndsi = image.normalizedDifference(['SR_B3', 'SR_B6']).rename('NDSI')
    bright = ndsi.gt(0.4)
    steep = slope_deg.gt(20)

    c_corr = c_correction(image, correction_region)
    m_corr = minnaert_correction(image, correction_region)
    use_c = bright.And(steep.Not())

    def blend_band(band_name):
        band_name = ee.String(band_name)
        c_band = c_corr.select([band_name.cat('_ccorr')])
        m_band = m_corr.select([band_name.cat('_ccorr')])
        return m_band.where(use_c, c_band).rename(band_name.cat('_ccorr'))

    hybrid = ee.ImageCollection.fromImages(ee.List(bands).map(blend_band)).toBands()
    expected = [f'{b}_ccorr' for b in bands]
    hybrid = hybrid.rename(expected).clamp(0, 1)

    return image.addBands(hybrid, None, True)


def otsu_threshold(img, band, region_geom):
    histogram = (
        img.select(band)
        .reduceRegion(
            reducer=ee.Reducer.histogram(),
            geometry=region_geom,
            scale=30,
            bestEffort=True,
            maxPixels=1e13,
        )
        .get(band)
    )

    hist = ee.Dictionary(histogram)
    counts = ee.Array(hist.get('histogram'))
    means = ee.Array(hist.get('bucketMeans'))
    size = means.length().get([0])
    total = counts.reduce(ee.Reducer.sum(), [0]).get([0])
    total_sum = means.multiply(counts).reduce(ee.Reducer.sum(), [0]).get([0])
    mean = total_sum.divide(total)

    indices = ee.List.sequence(1, size)

    def calc_bss(i):
        i = ee.Number(i)
        a_counts = counts.slice(0, 0, i)
        a_count = a_counts.reduce(ee.Reducer.sum(), [0]).get([0])
        a_means = means.slice(0, 0, i)
        a_mean = (
            a_means.multiply(a_counts)
            .reduce(ee.Reducer.sum(), [0])
            .get([0])
            .divide(a_count)
        )

        b_count = total.subtract(a_count)
        b_mean = total_sum.subtract(a_count.multiply(a_mean)).divide(b_count)

        return a_count.multiply(a_mean.subtract(mean).pow(2)).add(
            b_count.multiply(b_mean.subtract(mean).pow(2))
        )

    bss = indices.map(calc_bss)
    return means.sort(bss).get([-1])


def safe_otsu_from_band(img, band, region, scale=90, fallback=0.45):
    h = (
        img.select(band)
        .reduceRegion(
            reducer=ee.Reducer.histogram(maxBuckets=256),
            geometry=region,
            scale=scale,
            bestEffort=True,
            maxPixels=1e13,
        )
        .get(band)
    )

    has_hist = ee.Number(ee.Algorithms.If(h, 1, 0))
    total = ee.Number(
        ee.Algorithms.If(
            has_hist,
            ee.Array(ee.Dictionary(h).get('histogram'))
            .reduce(ee.Reducer.sum(), [0])
            .get([0]),
            0,
        )
    )
    ok = total.gt(0)

    return ee.Number(
        ee.Algorithms.If(ok, otsu_threshold(img, band, region), fallback)
    )


def adaptive_snow_threshold(img, band, region, scale=90):
    s = img.select(band)

    snow_frac = ee.Number(
        s.gt(0.5)
        .reduceRegion(
            reducer=ee.Reducer.mean(),
            geometry=region,
            scale=scale,
            bestEffort=True,
            maxPixels=1e13,
        )
        .get(band)
    )
    snow_frac = ee.Number(ee.Algorithms.If(snow_frac, snow_frac, 0))

    p = ee.Dictionary(
        s.reduceRegion(
            reducer=ee.Reducer.percentile([10, 50, 90]),
            geometry=region,
            scale=scale,
            bestEffort=True,
            maxPixels=1e13,
        )
    )

    p10 = ee.Number(ee.Algorithms.If(p.get(f'{band}_p10'), p.get(f'{band}_p10'), 0.25))
    p50 = ee.Number(ee.Algorithms.If(p.get(f'{band}_p50'), p.get(f'{band}_p50'), 0.45))
    p90 = ee.Number(ee.Algorithms.If(p.get(f'{band}_p90'), p.get(f'{band}_p90'), 0.65))

    otsu_thr = safe_otsu_from_band(img, band, region, scale, p50).clamp(p10, p90)

    return ee.Number(
        ee.Algorithms.If(
            snow_frac.gt(0.9),
            0.45,
            ee.Algorithms.If(snow_frac.lt(0.1), 0.30, otsu_thr),
        )
    )


def add_quality_bands(image):
    image = ee.Image(image)

    green = image.select('SR_B3_ccorr')
    swir1 = image.select('SR_B6_ccorr')
    swir2 = image.select('SR_B7_ccorr')
    red = image.select('SR_B4_ccorr')
    nir = image.select('SR_B5_ccorr')

    ndvi = red.subtract(nir).divide(red.add(nir)).clamp(-1, 1).rename('NDVI')
    ndsi = green.subtract(swir1).divide(green.add(swir1)).clamp(-1, 1).rename('NDSI')
    ndgi = green.subtract(red).divide(green.add(red)).clamp(-1, 1).rename('NDGI')
    ndwi = green.subtract(nir).divide(green.add(nir)).clamp(-1, 1).rename('NDWI')
    csi = nir.divide(swir2).rename('CSI')
    andsi = csi.subtract(ndsi).divide(csi.add(ndsi)).rename('ANDSI')

    albedo = image.expression(
        '0.356*B2 + 0.130*B3 + 0.373*B4 + 0.085*B5 + 0.072*B6 + 0.072*B7',
        {
            'B2': image.select('SR_B2_ccorr'),
            'B3': image.select('SR_B3_ccorr'),
            'B4': image.select('SR_B4_ccorr'),
            'B5': image.select('SR_B5_ccorr'),
            'B6': image.select('SR_B6_ccorr'),
            'B7': image.select('SR_B7_ccorr'),
        },
    ).rename('Albedo')

    vis = (
        image.select('SR_B2_ccorr')
        .add(image.select('SR_B3_ccorr'))
        .add(image.select('SR_B4_ccorr'))
        .divide(3)
        .rename('VIS')
    )

    snow_score = (
        ndsi.multiply(0.7)
        .add(vis.unitScale(0.05, 0.6).multiply(0.3))
        .rename('snowScore')
    )

    ndsi_score = ndsi.max(0).min(1)
    shadow = image.select('shadow')
    shadow_score = shadow.Not().rename('shadowScore')
    tb = image.select('TB')
    tb_norm = tb.unitScale(270, 320).rename('tbNorm')

    quality = (
        ndsi_score.multiply(0.6)
        .add(shadow_score.multiply(0.3))
        .add(tb_norm.multiply(0.1))
        .rename('quality')
    )

    return image.addBands([ndsi, andsi, ndgi, ndwi, albedo, snow_score, quality])

In [7]:
# -----------------------------------------------------------------------------
# PER-GLACIER PROCESSING
# -----------------------------------------------------------------------------


def process_glacier(feat):
    feat = ee.Feature(feat)

    min_footprint_cover = 0.50
    min_clear_observed = 0.50
    min_usable_cover = 0.50

    code_glacier = feat.get('COD_GLA')
    classifica = feat.get('CLASIFICA')
    glacier_geom = feat.geometry()

    landsat_89 = (
        L8.filterBounds(glacier_geom)
        .filterDate(START, END)
        .merge(L9.filterBounds(glacier_geom).filterDate(START, END))
        .map(set_tiles)
        .sort('system:time_start')
    )

    landsat_57 = (
        L7.filterBounds(glacier_geom)
        .filterDate(START, END)
        .merge(L5.filterBounds(glacier_geom).filterDate(START, END))
        .map(set_tiles)
        .sort('system:time_start')
    )

    # Determine the most frequently observed WRS row for this glacier.
    path_counts = ee.Dictionary(landsat_89.aggregate_histogram('WRS_ROW'))
    keys = path_counts.keys()
    values = path_counts.values()
    max_count = ee.Number(values.reduce(ee.Reducer.max()))
    max_index = values.indexOf(max_count)
    most_common_row = ee.Number.parse(ee.String(keys.get(max_index)))

    landsat_89 = landsat_89.filter(ee.Filter.eq('WRS_ROW', most_common_row))
    landsat_57 = landsat_57.filter(ee.Filter.eq('WRS_ROW', most_common_row))

    processed = landsat_89.map(scale_sr)
    processed2 = landsat_57.map(conversion)
    processed = processed.merge(processed2).sort('system:time_start')

    processed = (
        processed.map(lambda img: add_soft_cloud_mask(img, glacier_geom))
        .filter(ee.Filter.gte('footprint_cover', min_footprint_cover))
        .filter(ee.Filter.gte('clear_fraction_observed', min_clear_observed))
        .filter(ee.Filter.gte('usable_cover', min_usable_cover))
    )

    processed = (
        processed.map(topo_shadow)
        .map(add_topo_bands)
        .map(lambda img: hybrid_topo_correction(img, glacier_geom))
        .map(add_quality_bands)
        .sort('system:time_start')
    )

    glacier_area_m2 = ee.Number(
        ee.Image.pixelArea()
        .rename('area')
        .reduceRegion(
            reducer=ee.Reducer.sum(),
            geometry=glacier_geom,
            scale=30,
            maxPixels=1e13,
            tileScale=TILE_SCALE,
            bestEffort=True,
        )
        .get('area')
    )

    px_area = ee.Image.pixelArea()

    def summarize_image(img):
        img = ee.Image(img)
        region = glacier_geom.buffer(500)

        snow_score = img.select('snowScore')
        threshold = adaptive_snow_threshold(img, 'snowScore', region, 30)
        snow = snow_score.gt(threshold).rename('snow')
        snow_elev = dem.updateMask(snow)

        valid = snow_score.mask().rename('valid')
        snow_area = snow.selfMask().multiply(px_area).rename('snow_area_m2')
        valid_area = valid.selfMask().multiply(px_area).rename('valid_area_m2')

        alb_glacier = img.select('Albedo').updateMask(img.select('lit')).rename('alb')
        alb_snow_lit = (
            img.select('Albedo')
            .updateMask(img.select('lit'))
            .updateMask(snow)
            .rename('alb_snow')
        )
        lst = img.select('TB')

        stack = ee.Image.cat(
            [
                alb_glacier,
                alb_snow_lit,
                lst,
                snow_area,
                valid_area,
                snow_elev,
                snow_elev,
            ]
        )

        reducer = (
            ee.Reducer.median()
            .combine(
                reducer2=ee.Reducer.median(),
                outputPrefix='alb_snow_',
                sharedInputs=False,
            )
            .combine(
                reducer2=ee.Reducer.median(),
                outputPrefix='lst_',
                sharedInputs=False,
            )
            .combine(
                reducer2=ee.Reducer.sum(),
                outputPrefix='snow_',
                sharedInputs=False,
            )
            .combine(
                reducer2=ee.Reducer.sum(),
                outputPrefix='valid_',
                sharedInputs=False,
            )
            .combine(
                reducer2=ee.Reducer.median(),
                outputPrefix='snoele_',
                sharedInputs=False,
            )
            .combine(
                reducer2=ee.Reducer.mean(),
                outputPrefix='snoele_',
                sharedInputs=False,
            )
        )

        stats = stack.reduceRegion(
            reducer=reducer,
            geometry=glacier_geom,
            scale=30,
            bestEffort=True,
            maxPixels=1e13,
            tileScale=TILE_SCALE,
        )

        alb_med_glacier = stats.get('median')
        alb_med_snow = stats.get('alb_snow_median')

        # Keep nullable results as server-side objects until final filtering.
        therm = stats.get('lst_median')
        snow_sum_raw = stats.get('snow_sum')
        valid_sum_raw = stats.get('valid_sum')
        ele_median = stats.get('snoele_median')
        ele_mean = stats.get('snoele_mean')

        snow_sum = ee.Number(ee.Algorithms.If(snow_sum_raw, snow_sum_raw, 0))
        valid_sum = ee.Number(ee.Algorithms.If(valid_sum_raw, valid_sum_raw, 0))

        frac_valid = ee.Algorithms.If(
            valid_sum.gt(0),
            snow_sum.divide(valid_sum),
            None,
        )

        return ee.Feature(
            None,
            {
                'system:time_start': img.get('system:time_start'),
                'date': ee.Date(img.get('system:time_start')).format('YYYY-MM-dd'),
                'thresh': threshold,
                'albedoGlacier': alb_med_glacier,
                'albedoSnow': alb_med_snow,
                'snow_area_ha': snow_sum.divide(10000),
                'valid_area_ha': valid_sum.divide(10000),
                'fraction_area': frac_valid,
                'fraction_area2': snow_sum.divide(glacier_area_m2),
                'lst': therm,
                'elev_med': ele_median,
                'elev_mean': ele_mean,
                'code_glacier': code_glacier,
                'clasifica': classifica,
                'footprint_cover': img.get('footprint_cover'),
                'clear_fraction_observed': img.get('clear_fraction_observed'),
                'usable_cover': img.get('usable_cover'),
                'lst_cover': img.get('lst_cover'),
            },
        )

    return (
        ee.FeatureCollection(processed.map(summarize_image))
        .filter(ee.Filter.notNull(['albedoGlacier', 'fraction_area']))
        .filter(ee.Filter.gte('valid_area_ha', 5))
        .sort('system:time_start')
    )

In [8]:
# Mapping a function that returns FeatureCollections and flattening mirrors the
# final JavaScript pattern.
to_drive = glaciars.map(process_glacier)
out_fc = ee.FeatureCollection(to_drive).flatten()


In [9]:
# -----------------------------------------------------------------------------
# EXPORT
# -----------------------------------------------------------------------------
task = ee.batch.Export.table.toDrive(
    collection=out_fc,
    description='glacier_series2x1',
    fileNamePrefix='glacier_series2x1',
    fileFormat='CSV',
)
task.start()


print('Export task started.')
print('Task ID:', task.id)
print('Task status:', task.status())

/usr/local/lib/python3.12/dist-packages/ee/data.py:335: UserWarning: Your project has exceeded the compute quota of its noncommercial tier and is currently in restricted mode.
  warnings.warn(


Export task started.
Task ID: PWWCNNNK2NT4SOQKYVNFV2AU
Task status: {'state': 'READY', 'description': 'glacier_series2x1', 'priority': 100, 'creation_timestamp_ms': 1786554082430, 'update_timestamp_ms': 1786554082430, 'start_timestamp_ms': 0, 'task_type': 'EXPORT_FEATURES', 'id': 'PWWCNNNK2NT4SOQKYVNFV2AU', 'name': 'projects/upbeat-imprint-269809/operations/PWWCNNNK2NT4SOQKYVNFV2AU'}
